# project_12_biosensor — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — biosensor architectures, analyte + readout, sensor metrics

**Standard slot:** *define & explore.* **For Project 12 this means:** survey the two switch
architectures (allosteric / LOCKR-style conformational switches **and** split-reporter systems),
**choose your analyte and your readout**, write down the binder **and** sensor metrics + cutoffs, and
run a deterministic **mock** mini-run (binder → switch → ON/OFF) as your "hello-world" (D0).

Run `00_setup.ipynb` first in this session. A real binder campaign + switch modeling wants an
**A100** (see `MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the
plumbing anywhere, then switch to the real backends on Colab Pro / A100.

> This project builds on the **binder-family workflow** (Project 06): the binder module is the same
> two-paradigm campaign. The new part is the **switch / split-reporter** that turns *binding* into
> *signal* — and reasoning about whether that signal has a usable **dynamic range**.

## Biosensor architectures (pick one to build on)

| Family | How binding → signal | Reporter | Key reference |
|--------|----------------------|----------|---------------|
| **Split-reporter** | binding reconstitutes a split enzyme/fluorophore | split-luciferase / **NanoBiT** (luminescence), split-FP (**FRET**) | Dixon 2016; Quijano-Rubio 2021 |
| **Conformational switch (LOCKR)** | analyte (or an exposed "key") displaces a **latch**, releasing a functional/reporter element | de novo cage + latch + key | Langan 2019 |

Both turn a **binding event** into a **measurable change**. The design decision that matters is
**mechanical coupling**: the binder must engage the analyte in a way that *toggles* the switch. A
great binder bolted to a switch that never moves is not a sensor.

## The metrics, precisely — binder metrics **and** sensor metrics

**Binder metrics** (same as the binder family; drive `design_type="binder"` filtering):

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the binder | thermostability / ΔG |
| **pae_interaction** | Å | AF2-Multimer error across the **binder–analyte interface** (key binder metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |

**Sensor metrics** (new — describe the *transduction*, not just binding):

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| on_signal / off_signal | a.u. | modeled signal **with / without** analyte | a measured luminescence/FRET value |
| **dynamic_range** | fold | `on_signal / off_signal` — the headline sensor metric | a measured limit of detection |
| toggle_score | 0–1 | modeled separation between OFF and ON state conformations | that the real switch flips |
| background_leak | 0–1 | OFF-state signal leak (lower better) | assay background in the lab |

> The binder cutoffs are the shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80,
> pae_interaction ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6.** The **sensor** is judged separately on
> **dynamic range** — and there is a real **affinity-vs-dynamic-range trade-off** (a too-tight binder
> can lock the switch ON regardless of analyte). A passing design is a **hypothesis** until a
> functional luminescence/FRET dose-response is run.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Choose the analyte + readout (STUDENT CHOICE)

**You pick the analyte.** Good capstone choices are small, structurally-characterized protein
biomarkers — e.g. a **cytokine** (inflammation) or a **cardiac marker** (e.g. a troponin subunit).
The catalog deliberately leaves this open: choose something with a verified RCSB structure and a clear
point-of-care motivation, and **mark the accession "candidate — verify on RCSB"** in Week 1
(`data/README.md`). Then pick a **readout**: luminescence (split-luciferase/NanoBiT) or FRET
(split-fluorophore).

Below we just *declare* an EXAMPLE analyte + epitope so the notebook runs end-to-end; **replace them
with your verified choice** (numbering depends on the PDB you clean).

In [ ]:
import biosensor_tools as bt

# --- STUDENT CHOICE: set these in Week 1 from your verified analyte structure ---
ANALYTE  = "ANALYTE"                 # e.g. your chosen cytokine / cardiac marker (cleaned target PDB)
# EXAMPLE epitope residues on the analyte — VERIFY/REPLACE from the analyte structure (data/README.md).
HOTSPOTS = bt.parse_hotspots("A12,A45,A60")   # EXAMPLE_DATA placeholder residues
READOUT  = "split_luciferase"        # one of: split_luciferase | nanobit | split_fluorophore_fret
SWITCH_FAMILY = "split_reporter"     # "split_reporter" (luminescence/FRET) or "lockr" (cage+latch)

print("analyte      :", ANALYTE, " (STUDENT CHOICE — verify the accession on RCSB)")
print("epitope/hotspots:", HOTSPOTS, " (EXAMPLE — replace with your verified residues)")
print("readout      :", READOUT)
print("switch family:", SWITCH_FAMILY)

## 2 · Mock hello-world: binder → switch → ON/OFF

`scripts/biosensor_tools.py` exposes the binder paradigms (`generate_binders_bindcraft`,
`generate_binders_rfdiffusion`, `af2_multimer`) **plus** the switch module (`design_switch`,
`integrate_binder_switch`, `model_two_state` / `two_state_readout`). The **mock** backend is
deterministic and GPU-free so you can develop the whole binding→signal plumbing. **Never report mock
numbers as real** — they are `SYNTHETIC` by construction (no real luminescence, no real LOD).

In [ ]:
# A few binders from each paradigm, scored by mock AF2-Multimer. All numbers are SYNTHETIC.
bc = bt.generate_binders_bindcraft(ANALYTE, HOTSPOTS, n=3, tool="mock")
rf = bt.generate_binders_rfdiffusion(ANALYTE, HOTSPOTS, n=3, tool="mock")
bt.score_designs(bc, tool="mock")
bt.score_designs(rf, tool="mock")

b = bc[0]
print("example binder design:")
print("  id   :", b.design_id, " len:", b.length, "aa")
print("  pae_interaction =", b.pae_interaction, " scrmsd =", b.scrmsd,
      " sc =", b.shape_complementarity, " (SYNTHETIC)")

In [ ]:
# Design a switch, integrate the binder, and reason about ON/OFF (all SYNTHETIC).
switch = bt.design_switch(scaffold="rfdiff_scaffold_01", reporter=READOUT,
                          family=SWITCH_FAMILY, n=1, tool="mock")[0]
construct = bt.integrate_binder_switch(b, switch, tool="mock")
bt.two_state_readout(construct, tool="mock")   # fills on/off/dynamic_range

print("switch   :", switch.switch_id, " toggle_score=", switch.toggle_score,
      " background_leak=", switch.background_leak, " (SYNTHETIC)")
print("construct:", construct.construct_id)
print("  on_signal =", construct.on_signal, " off_signal =", construct.off_signal,
      " dynamic_range =", construct.dynamic_range, " (SYNTHETIC)")
print("  LOD planning flag:", bt.estimate_lod(construct.dynamic_range, assay_cv=0.10))
print("\nReminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.")
print("dynamic_range here is SYNTHETIC — a real value needs a luminescence/FRET dose-response (nb 05).")

## 3 · Epitope-coverage proxy (does the binder engage the intended epitope?)

For a biosensor the binder must engage the analyte at the epitope you chose, so that binding couples
to the switch. `hotspot_overlap()` is a geometry proxy (fraction of the chosen epitope contacted) — a
teaching stand-in. Higher ⇒ the binder is engaging where you intended (not a guarantee of switching).

In [ ]:
for d in bc[:3]:
    ov = bt.hotspot_overlap(d.contact_residues, HOTSPOTS)
    print(f"{d.design_id}: contacts {d.contact_residues} -> epitope coverage = {ov} (SYNTHETIC)")

## Visualize a binder–analyte complex (py3Dmol)

Use this to eyeball a predicted binder–analyte complex (or, later, the integrated construct) once you
have a real PDB from AF2-Multimer / two-state modeling.

In [ ]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer / two-state prediction writes a complex PDB):
# show_complex("results/af2/top_complex.pdb")
print("show_complex(pdb_path) ready.")

## D0 checklist
- [ ] Analyte chosen (small protein biomarker) + accession marked **candidate — verify on RCSB**.
- [ ] Readout chosen (split-luciferase / NanoBiT luminescence, or split-FP FRET) + switch family.
- [ ] Epitope/hotspots derived from the analyte structure (not invented).
- [ ] One-paragraph definition of each **binder** and **sensor** metric **with** its "does not mean" note.
- [ ] Reproduced mock hello-world (binder → switch → ON/OFF) with metrics printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria (e.g. target dynamic range) + controls
      (no-analyte, off-target); `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the binder campaign + the switch module.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — binder module + switch module

**Standard slot:** *design campaign.* **For Project 12 this is the core:** run the **binder module**
(the Project-06 two-paradigm workflow) against your analyte epitope, **and** design/borrow the
**switch module** (an RFdiffusion scaffold or a LOCKR-style cage), then write the results CSVs (D2):
- **Binder — BindCraft** (one-shot hallucination, AF2-Multimer in the loop) — **50–200** designs.
- **Binder — RFdiffusion binder mode → ProteinMPNN** — **500–1000** backbones → sequences.
- **Switch** — a handful of switch scaffolds (split-reporter and/or LOCKR-style), scored on their
  intrinsic toggle quality.

> **Compute honesty:** a real binder campaign + switch modeling at this scale wants an **A100**
> (Colab Pro+ or a cluster). Free **T4** = a *small fallback* (FreeBindCraft, small `num_designs`, a
> small RFdiffusion batch + ESMFold triage). The cells below run on the deterministic **mock** backend
> so the plumbing executes anywhere; the real calls + A100 notes are shown alongside. Run
> `00_setup.ipynb` first.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change!)

The binder and switch tools live in fast-moving upstream repos. **Pin commits** and **verify the URLs
still exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin and
log it). The generation itself needs an A100; this check needs nothing.

In [ ]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   BindCraft      https://github.com/martinpacesa/BindCraft        # binder paradigm #1; pin <commit>
#   FreeBindCraft  https://github.com/cytokineking/FreeBindCraft     # free-tier fallback — VERIFY it exists; pin <commit>
#   RFdiffusion    https://github.com/RosettaCommons/RFdiffusion     # binder mode + scaffold/switch; pin <commit>
#   ColabDesign    https://github.com/sokrypton/ColabDesign          # RFdiffusion-binder + ProteinMPNN; pin <commit>
#   ColabFold      https://github.com/sokrypton/ColabFold            # AF2-Multimer / two-state modeling; pin <commit>
PINNED = {
    "BindCraft":     "https://github.com/martinpacesa/BindCraft",
    "FreeBindCraft": "https://github.com/cytokineking/FreeBindCraft",
    "RFdiffusion":   "https://github.com/RosettaCommons/RFdiffusion",
    "ColabDesign":   "https://github.com/sokrypton/ColabDesign",
    "ColabFold":     "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:14s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:14s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.")
print("Switch literature (no install): Langan 2019 LOCKR; Quijano-Rubio 2021 biosensors; Dixon 2016 NanoBiT.")

## 1 · Define the campaign

Same analyte + epitope as notebook 01. Set honest campaign sizes; the cells run on `mock` so they
execute anywhere. On Colab (A100) switch `TOOL_*` to the real backends — and **shrink the numbers on
a T4** (FreeBindCraft, a small RFdiffusion batch).

In [ ]:
import biosensor_tools as bt
import pandas as pd

ANALYTE  = "ANALYTE"                          # STUDENT CHOICE — your verified biomarker target
HOTSPOTS = bt.parse_hotspots("A12,A45,A60")   # EXAMPLE — replace with your verified epitope residues
READOUT  = "split_luciferase"                 # split_luciferase | nanobit | split_fluorophore_fret
SWITCH_FAMILY = "split_reporter"              # "split_reporter" or "lockr"

# Honest campaign sizes (catalog): BindCraft 50-200, RFdiffusion 500-1000 backbones.
# We use small mock counts here so the dry run is fast; scale up with the real backend on A100.
N_BINDCRAFT   = 60      # -> 50-200 on A100; fewer (FreeBindCraft) on T4
N_RFDIFFUSION = 200     # -> 500-1000 backbones on A100; small batch on T4
N_SWITCH      = 8       # a handful of switch scaffolds to compare architectures

TOOL_BINDCRAFT   = "mock"   # -> "bindcraft" / "freebindcraft" on Colab
TOOL_RFDIFFUSION = "mock"   # -> "rfdiffusion" on Colab
TOOL_AF2         = "mock"   # -> "af2" (ColabFold AF2-Multimer) on Colab
TOOL_SWITCH      = "mock"   # -> "rfdiffusion" (scaffold) or "lockr" (borrow a cage) on Colab

print(f"BindCraft   : n={N_BINDCRAFT}  tool={TOOL_BINDCRAFT}")
print(f"RFdiffusion : n={N_RFDIFFUSION} tool={TOOL_RFDIFFUSION}")
print(f"Switch      : n={N_SWITCH} family={SWITCH_FAMILY} reporter={READOUT} tool={TOOL_SWITCH}")
print("analyte/epitope:", ANALYTE, HOTSPOTS)

## 2 · Binder paradigm #1 — BindCraft campaign

One-shot hallucination with AF2-Multimer in the loop. On A100 this produces 50–200 binders
pre-filtered on interface confidence; we still re-score with AF2-Multimer so the head-to-head with
RFdiffusion is apples-to-apples. The `mock` backend returns deterministic `SYNTHETIC` designs.

In [ ]:
# Real call (Colab, A100): bt.generate_binders_bindcraft(ANALYTE, HOTSPOTS, n=N_BINDCRAFT, tool="bindcraft")
#   free-tier fallback: tool="freebindcraft", smaller N. See MANUAL.md §2 / scripts/biosensor_tools.py TODOs.
bindcraft = bt.generate_binders_bindcraft(ANALYTE, HOTSPOTS, n=N_BINDCRAFT, tool=TOOL_BINDCRAFT)
bt.score_designs(bindcraft, tool=TOOL_AF2)     # AF2-Multimer -> pae_interaction, plddt, scrmsd, sc
print(f"BindCraft pool: {len(bindcraft)} designs (tool={TOOL_BINDCRAFT}; SYNTHETIC if mock)")
print("example:", bindcraft[0].design_id, "pae_interaction=", bindcraft[0].pae_interaction)

## 3 · Binder paradigm #2 — RFdiffusion binder campaign → ProteinMPNN

Diffuse binder backbones docked at the epitope, then ProteinMPNN designs sequences, then AF2-Multimer
re-predicts each complex. On A100 this is 500–1000 backbones (the per-backbone hit rate is low — that
is normal). The `mock` backend stands in for the whole chain.

In [ ]:
# Real call (Colab, A100): bt.generate_binders_rfdiffusion(ANALYTE, HOTSPOTS, n=N_RFDIFFUSION,
#   tool="rfdiffusion", mpnn_temperature=0.1, num_seq_per_backbone=8). AF2-Multimer is the slow step.
rfdiff = bt.generate_binders_rfdiffusion(ANALYTE, HOTSPOTS, n=N_RFDIFFUSION, tool=TOOL_RFDIFFUSION)
bt.score_designs(rfdiff, tool=TOOL_AF2)
print(f"RFdiffusion pool: {len(rfdiff)} designs (tool={TOOL_RFDIFFUSION}; SYNTHETIC if mock)")
print("example:", rfdiff[0].design_id, "pae_interaction=", rfdiff[0].pae_interaction)

## 4 · Switch module — design/borrow the transduction element

The switch converts binding into signal. Two routes (see `MANUAL.md §1`):
- **split-reporter:** an RFdiffusion scaffold presenting a split-luciferase/NanoBiT or split-FP FRET
  pair that reconstitutes on binding;
- **LOCKR-style cage:** borrow/adapt a published de novo cage+latch (Langan 2019), graft the
  reporter/functional element, redesign the latch so the analyte displaces it.

We score each switch on its **intrinsic** toggle quality (state separation + OFF-state leak) — this is
analyte-independent; the binder-coupled ON/OFF behaviour comes in notebook 04. Mock numbers are
SYNTHETIC.

In [ ]:
# Real call (Colab, A100): bt.design_switch(scaffold=..., reporter=READOUT, family=SWITCH_FAMILY,
#   n=N_SWITCH, tool="rfdiffusion")  # or tool="lockr" to borrow/adapt a published LOCKR cage.
switches = bt.design_switch(scaffold="rfdiff_scaffold", reporter=READOUT,
                            family=SWITCH_FAMILY, n=N_SWITCH, tool=TOOL_SWITCH)
# Also compare a LOCKR-style family so notebook 04's architecture benchmark has two architectures.
switches_lockr = bt.design_switch(scaffold="lockr_cage", reporter="",
                                  family="lockr", n=N_SWITCH, tool=TOOL_SWITCH)
print(f"switch pool: {len(switches)} {SWITCH_FAMILY} + {len(switches_lockr)} lockr (SYNTHETIC if mock)")
s = switches[0]
print("example switch:", s.switch_id, "toggle_score=", s.toggle_score, "background_leak=", s.background_leak)

## 5 · Assemble + persist the pools

Write one tidy CSV per binder paradigm (plus a combined one) and one for the switches. These feed
notebook 03 (the shared **binder** filter) and notebook 04 (integration + ON/OFF). We add an EXAMPLE
physics column (`rosetta_dG`) here so the binder physics layer has something to act on in the dry run
— on Colab these come from FreeBindCraft/PyRosetta; for `mock` they are SYNTHETIC.

In [ ]:
import pandas as pd

def binder_pool_to_df(designs):
    rows = []
    for d in designs:
        # Mock dry run: attach an EXAMPLE_DATA interface energy so Layer 3 (physics) is exercised.
        # On Colab, replace with the real FreeBindCraft/PyRosetta rosetta_dG + solubility.
        rdg = -45.0 + (bt._hashints("dG", d.design_id) % 40)   # SYNTHETIC, range ~ -45..-6 REU
        rows.append(dict(
            design_id=d.design_id, paradigm=d.paradigm, target=d.target,
            length=d.length, sequence=d.sequence,
            plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
            shape_complementarity=d.shape_complementarity,
            rosetta_dG=round(float(rdg), 2), solubility=0.3,
            contact_residues=",".join(d.contact_residues),
            epitope_coverage=bt.hotspot_overlap(d.contact_residues, d.hotspots),
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

def switch_pool_to_df(switches):
    return pd.DataFrame([dict(
        switch_id=s.switch_id, family=s.family, reporter=s.reporter, scaffold=s.scaffold,
        closed_plddt=s.closed_plddt, open_plddt=s.open_plddt,
        toggle_score=s.toggle_score, background_leak=s.background_leak, synthetic=s.synthetic,
    ) for s in switches])

df_bc = binder_pool_to_df(bindcraft); df_bc.to_csv("results/bindcraft_designs.csv", index=False)
df_rf = binder_pool_to_df(rfdiff);    df_rf.to_csv("results/rfdiffusion_designs.csv", index=False)
combined = pd.concat([df_bc, df_rf], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

df_sw = switch_pool_to_df(list(switches) + list(switches_lockr))
df_sw.to_csv("results/switch_designs.csv", index=False)

print("wrote results/bindcraft_designs.csv   ", df_bc.shape)
print("wrote results/rfdiffusion_designs.csv ", df_rf.shape)
print("wrote results/all_designs.csv         ", combined.shape)
print("wrote results/switch_designs.csv      ", df_sw.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.")
combined.head(4)

## D2 checklist
- [ ] BindCraft binder pool generated at honest scale (50–200 on A100; FreeBindCraft/small on T4).
- [ ] RFdiffusion-binder pool generated (500–1000 backbones → ProteinMPNN on A100).
- [ ] Every binder scored by AF2-Multimer (`pae_interaction` parsed); both pools written to `results/`.
- [ ] Switch module: ≥1 architecture designed/borrowed (split-reporter and/or LOCKR), `results/switch_designs.csv`.
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured; 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared binder filter** on the binder pools.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 12** you build `fp.Design` **binder** objects from both binder pools, call
`fp.run_pipeline(..., design_type="binder")`, and `fp.report(...)` the survival funnel + ranked CSV,
**per paradigm** so the head-to-head is fair (D3 part 1). The binder is the recognition element of the
sensor — it must pass the binder bar *before* you couple it to a switch.

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` exist.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"binder"` cutoffs: scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

## Build `Design` (binder) objects from the pools

Map each binder-pool row onto `fp.Design` with `design_type="binder"`. The binder metrics drive the
layers: `scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency), and
`rosetta_dG`/`shape_complementarity`/`solubility` (Layer 3 physics). We keep `paradigm` +
`epitope_coverage` in `extra` for the integration + ON/OFF analysis in notebook 04. (Mock has no
independent orthogonal predictor, so we run Layers 1+3 here; on Colab add a second predictor for
Layer 2.)

In [ ]:
import os
import pandas as pd

# Regenerate the binder pools if a fresh session lost them (deterministic mock).
if not (os.path.exists("results/bindcraft_designs.csv") and os.path.exists("results/rfdiffusion_designs.csv")):
    import biosensor_tools as bt
    ANALYTE, HOTSPOTS = "ANALYTE", bt.parse_hotspots("A12,A45,A60")
    bc = bt.generate_binders_bindcraft(ANALYTE, HOTSPOTS, n=60, tool="mock");  bt.score_designs(bc, tool="mock")
    rf = bt.generate_binders_rfdiffusion(ANALYTE, HOTSPOTS, n=200, tool="mock"); bt.score_designs(rf, tool="mock")
    def _q(designs, p):
        rows=[dict(design_id=d.design_id, paradigm=d.paradigm, length=d.length, sequence=d.sequence,
                   plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
                   shape_complementarity=d.shape_complementarity,
                   rosetta_dG=round(-45.0+(bt._hashints("dG",d.design_id)%40),2), solubility=0.3,
                   epitope_coverage=bt.hotspot_overlap(d.contact_residues,d.hotspots), synthetic=d.synthetic)
              for d in designs]
        pd.DataFrame(rows).to_csv(p, index=False)
    _q(bc, "results/bindcraft_designs.csv"); _q(rf, "results/rfdiffusion_designs.csv")

df_bc = pd.read_csv("results/bindcraft_designs.csv")
df_rf = pd.read_csv("results/rfdiffusion_designs.csv")

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3),
        extra={"paradigm": r.get("paradigm"), "epitope_coverage": r.get("epitope_coverage")},
    )

binders_bc = [row_to_binder(r) for _, r in df_bc.iterrows()]
binders_rf = [row_to_binder(r) for _, r in df_rf.iterrows()]
print(f"built {len(binders_bc)} BindCraft + {len(binders_rf)} RFdiffusion binder Designs")

## Run the pipeline — per paradigm (fair head-to-head)

`run_pipeline(design_type="binder")` applies the binder cutoffs in order and returns a ranked
DataFrame with survival counts in `df.attrs`. We run **each paradigm separately** so the
survival-at-each-layer funnels are comparable. We use Layers 1+3 here (mock has no independent
orthogonal source; add Layer 2 on Colab with a second predictor).

In [ ]:
def run_one(designs, label):
    df = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
    df["paradigm"] = label
    surv = df.attrs["survival"]; n = df.attrs["n_total"]
    passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: {n} designs, survival {surv}, all-layers hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")
    return df

ranked_bc = run_one(binders_bc, "bindcraft")
ranked_rf = run_one(binders_rf, "rfdiffusion")

ranked = pd.concat([ranked_bc, ranked_rf], ignore_index=True).sort_values(
    ["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/all_ranked.csv", index=False)
print("\nwrote results/all_ranked.csv", ranked.shape)
ranked.head(10)[["design_id", "paradigm", "layers_passed", "score",
                 "scrmsd", "plddt", "pae_interaction", "rosetta_dG"]]

## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the
**combined** pool for one comparable figure; the per-paradigm runs above are the rigorous version.
Read the bars as a funnel: steep drops show which layer discriminates.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

all_binders = binders_bc + binders_rf
df_all = fp.run_pipeline(all_binders, design_type="binder", use_layers=(1, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p12")
print("\nsaved results/p12_survival.png + results/p12_ranked.csv")
top

## Honest hit-rate accounting (per paradigm)

Report `N passing all layers / N generated` for **each** paradigm — this is the number that decides
which binders are worth coupling to a switch in notebook 04. Remember: survival is *enrichment*, not
*correctness*, and even a passing binder is only the *recognition* half of a sensor. Mock numbers are
SYNTHETIC.

In [ ]:
for label, df in [("bindcraft", ranked_bc), ("rfdiffusion", ranked_rf)]:
    n = len(df); passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: layers_passed distribution {df['layers_passed'].value_counts().sort_index().to_dict()}")
    print(f"{'':12s}  all-layers survivors = {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")

## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Survival-at-each-layer reported **per paradigm** (funnel figure `results/p12_survival.png`).
- [ ] Honest hit-rate accounting (N pass / N generated) for BindCraft and RFdiffusion.
- [ ] Mapping assumptions (which fields → which `Design` attributes) written down.

**Next:** `04_validate.ipynb` — integrate binder + switch and reason about ON/OFF states.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — integrate binder + switch, reason about ON/OFF states

**Standard slot:** *validate (in silico).* **For Project 12 this is the core integration step:**
couple the top binders to the switch, **model the integrated construct**, and reason about the
**ON/OFF (two-state) behaviour** — with publication-style figures (D3 part 2). The benchmark here is
**switch architecture** (split-reporter vs LOCKR) and the **affinity-vs-dynamic-range** trade-off.

Two-state AF2 modeling of the switch is the `[extension]`: model the OFF (closed / split-apart) and
ON (open / reconstituted) conformations and derive a relative signal. The `mock` backend gives a
deterministic SYNTHETIC ON/OFF so the reasoning plumbing runs anywhere.

Needs `results/all_ranked.csv` (nb 03), `results/switch_designs.csv` + the binder pools (nb 02).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Integrate the top binders with the switch(es)

Take the top binder survivors (nb 03) and couple each to a switch (nb 02). `integrate_binder_switch()`
assembles the construct and carries the binder metrics; `two_state_readout()` then models the ON
(with analyte) and OFF (no analyte) signals and computes the **dynamic_range = on/off** — the headline
sensor metric. Mock numbers are SYNTHETIC (no real luminescence).

In [ ]:
import pandas as pd, numpy as np, os
import biosensor_tools as bt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
switches = pd.read_csv("results/switch_designs.csv")

# Rebuild lightweight binder objects for the top survivors (mock-safe; sequence carried in the pools).
pools = pd.concat([pd.read_csv("results/bindcraft_designs.csv"),
                   pd.read_csv("results/rfdiffusion_designs.csv")], ignore_index=True)
seq_by_id = pools.set_index("design_id")["sequence"].to_dict()

surv = ranked[ranked["layers_passed"] >= 3].copy()
TOP_N = 10
top_binders = (surv.sort_values(["score"], ascending=False)
                   .groupby("paradigm").head(TOP_N))
print(f"coupling {len(top_binders)} top binder survivors to switches (SYNTHETIC if mock)")

def mk_binder(row):
    b = bt.BinderDesign(design_id=str(row["design_id"]), sequence=str(seq_by_id.get(row["design_id"], "M")),
                        paradigm=str(row["paradigm"]), target="ANALYTE")
    b.plddt=row.get("plddt"); b.pae_interaction=row.get("pae_interaction"); b.scrmsd=row.get("scrmsd")
    b.shape_complementarity=row.get("shape_complementarity"); b.rosetta_dG=row.get("rosetta_dG")
    return b

In [ ]:
# Couple each top binder to one representative switch per architecture and model ON/OFF.
constructs = []
reps = switches.groupby("family").first().reset_index()   # one representative switch per family
for _, brow in top_binders.iterrows():
    binder = mk_binder(brow)
    for _, srow in reps.iterrows():
        sw = bt.SwitchDesign(switch_id=str(srow["switch_id"]), family=str(srow["family"]),
                             reporter=str(srow.get("reporter", "")), scaffold=str(srow.get("scaffold", "")),
                             toggle_score=srow.get("toggle_score"), background_leak=srow.get("background_leak"),
                             synthetic=True)
        c = bt.integrate_binder_switch(binder, sw, tool="mock")
        bt.two_state_readout(c, tool="mock")
        constructs.append(c)

cdf = pd.DataFrame([c.as_row() for c in constructs])
cdf.to_csv("results/constructs.csv", index=False)
print("wrote results/constructs.csv", cdf.shape)
cdf.head(6)[["construct_id", "family", "pae_interaction", "on_signal", "off_signal", "dynamic_range"]]

## 2 · Benchmark: switch architecture (split-reporter vs LOCKR)

Compare the two switch architectures on the **dynamic range** they deliver across the same set of top
binders. A fair comparison couples the *same* binders to each architecture and reports the
*distribution* of dynamic range, not the single best. Mock numbers are SYNTHETIC.

In [ ]:
summary = []
for fam, g in cdf.groupby("family"):
    summary.append(dict(family=fam, n=len(g),
                        median_dynamic_range=round(float(g["dynamic_range"].median()), 2),
                        max_dynamic_range=round(float(g["dynamic_range"].max()), 2),
                        median_off_signal=round(float(g["off_signal"].median()), 2)))
summary = pd.DataFrame(summary)
print("switch-architecture benchmark (SYNTHETIC if mock):")
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.6))
fams = list(cdf["family"].unique())
ax.boxplot([cdf[cdf["family"] == f]["dynamic_range"].dropna() for f in fams])
ax.set_xticks(range(1, len(fams) + 1)); ax.set_xticklabels(fams)
ax.set_ylabel("dynamic range (on/off, fold)"); ax.set_xlabel("switch architecture")
ax.set_title("Dynamic range by switch architecture (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p12_architecture.png", dpi=150); plt.show()
print("saved results/p12_architecture.png")

## 3 · The affinity-vs-dynamic-range trade-off

A real tension in biosensor design: a **too-tight** binder can lock the switch ON regardless of the
analyte (no switching), while a **too-weak** one never forms the ON state. Plot the binder metric
(`pae_interaction`, lower = more confident interface) against the modeled **dynamic range** to see the
trade-off. This is the figure that makes the project a *study*: the best *sensor* is not always the
best *binder*. Mock numbers are SYNTHETIC.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for fam, g in cdf.groupby("family"):
    ax.scatter(g["pae_interaction"], g["dynamic_range"], alpha=0.6, label=fam)
ax.set_xlabel("pae_interaction (Å, lower = more confident binder)")
ax.set_ylabel("dynamic range (on/off, fold)")
ax.set_title("Affinity proxy vs dynamic range (EXAMPLE_DATA if mock)")
ax.legend()
plt.tight_layout(); plt.savefig("results/p12_tradeoff.png", dpi=150); plt.show()
print("saved results/p12_tradeoff.png")
print("Read this as: the strongest binder is not automatically the best SENSOR — tune for dynamic range.")

## 4 · Two-state AF2 modeling of the switch `[extension]`

The rigorous version of ON/OFF: model the construct **without** analyte (OFF: split halves apart /
latch in place) and **with** analyte (ON: halves together / latch displaced), then derive a relative
signal from the predicted state populations + interface confidence. On Colab this uses AF2 multi-state
tricks (templates, state-specific MSAs) or a LOCKR cage+key model (A100). Here we scaffold where it
plugs in — the `mock` ON/OFF above stands in for it.

In [ ]:
# Scaffold: on Colab (A100), replace two_state_readout(tool="mock") with tool="af2":
#   for each construct, model OFF and ON conformations and parse a relative signal proxy.
# Pinned upstream (verify): https://github.com/sokrypton/ColabFold  (two-state via templates/MSAs).
# The output is a MODELED proxy, NOT measured luminescence — the assay (notebook 05) measures it.
print("Two-state AF2 modeling [extension]: switch tool='mock' -> 'af2' in two_state_readout on Colab (A100).")
print("These remain MODELED proxies; the real ON/OFF is the luminescence/FRET dose-response in nb 05.")

## 5 · Select the top integrated constructs

Rank constructs by **dynamic range** first (the sensor metric), with the binder confidence
(`pae_interaction`) as a tie-breaker, and save the shortlist for the validation plan (notebook 05).

In [ ]:
cdf["pae_rank"] = cdf["pae_interaction"].rank(ascending=True)   # lower pae = better binder
top_constructs = cdf.sort_values(["dynamic_range", "pae_rank"],
                                 ascending=[False, True]).head(20)
top_constructs.to_csv("results/top_constructs.csv", index=False)
print("wrote results/top_constructs.csv:", top_constructs.shape)
print("by architecture:", top_constructs.groupby("family").size().to_dict())
top_constructs.head(8)[["construct_id", "family", "pae_interaction", "dynamic_range", "off_signal"]]

## D3 (part 2) checklist
- [ ] Top binders coupled to the switch(es); `results/constructs.csv` with ON/OFF + dynamic range.
- [ ] Switch-architecture benchmark (split-reporter vs LOCKR) figure `results/p12_architecture.png`.
- [ ] Affinity-vs-dynamic-range trade-off figure `results/p12_tradeoff.png` + honest discussion.
- [ ] Two-state AF2 modeling `[extension]` wired up (or its scaffold documented for Colab).
- [ ] `results/top_constructs.csv`: ranked by dynamic range, ready for the validation plan.

**Next:** `05_validation_plan.ipynb` — the luminescence/FRET dose-response + LOD + controls.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — dose-response, LOD, controls, multiplexing

**Standard slot:** *validation plan.* **For Project 12 this means:** turn the top integrated
constructs into a **costed, controlled functional-readout plan** — a **luminescence/FRET
dose-response**, an **LOD estimate**, the mandatory controls (**no-analyte / blank** and
**off-target**), an expression strategy, and a **multiplexing** concept `[stretch]` (D4/D5).

A construct that looks good in silico is a **hypothesis** — the dose-response is what tests it. **No
fabricated LOD or luminescence numbers**: you estimate detectability from a *planned* dose-response
with replicates and a blank. Needs `results/top_constructs.csv` (notebook 04).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Draft the functional-readout validation plan

Generate a plan card from the top constructs: the dose-response assay, controls, expression, timeline,
costed reagents. Fill the `<...>` from your own numbers; this is the deliverable other people read.

In [ ]:
import pandas as pd, os

top = pd.read_csv("results/top_constructs.csv") if os.path.exists("results/top_constructs.csv") else pd.DataFrame()
n_top = len(top)
by_fam = top.groupby("family").size().to_dict() if n_top else {}

plan = f"""# De Novo Binder->Biosensor Validation Plan (Project 12 — by <your name>, <date>)

## Analyte + readout
Analyte: <your chosen biomarker> (verify RCSB accession). Readout: <split-luciferase/NanoBiT
luminescence, or split-FP FRET>. Switch family/families carried: {by_fam}.

## Candidates
Top {n_top} integrated constructs carried forward; see results/top_constructs.csv. EVERY in-silico
number is a HYPOTHESIS: pae_interaction is binder confidence (not affinity); dynamic_range is a
MODELED proxy (not measured signal); there is NO measured LOD yet.

## Expression strategy
- Construct (binder + switch fusion): E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC.
  For split-luciferase/NanoBiT, confirm the reporter folds and is active in the fusion context.
- Analyte/biomarker reagent: recombinant (mammalian/insect or commercial); confirm it is the right
  isoform/PTM state for your sensor.

## Functional assay (the point of the project): dose-response
1. Titrate the analyte across a wide concentration range (e.g. log-spaced, >= 8 points + blank).
2. Read the signal: luminescence (split-luciferase/NanoBiT) or FRET ratio (split-FP).
3. Fit signal vs [analyte] (e.g. 4-parameter logistic) -> EC50 + dynamic range (max/min, fold).
4. ESTIMATE the LOD from the fit: blank mean + 3*SD (or the lowest distinguishable dose). This is a
   MEASURED estimate from YOUR data — never a number copied from a model.

## Controls (MANDATORY)
- NO-ANALYTE / BLANK: buffer only -> defines the OFF/background signal and the LOD floor. The single
  most important control for a sensor (a leaky OFF state kills dynamic range).
- OFF-TARGET: a structurally-related but wrong analyte (or an unrelated protein) -> the sensor must
  NOT light up. This is the specificity control.
- POSITIVE: a known concentration of the true analyte (and, if available, an established sensor/ELISA)
  to confirm the assay works and to cross-calibrate.

## Realistic expectations
Coupling binding to a CLEAN ON/OFF signal is hard; dynamic range vs binder affinity is a real
trade-off; MOST integrated constructs need iteration (linker length/rigidity, latch redesign,
reporter placement). Report the dynamic range and LOD you MEASURE, honestly — including constructs
that don't switch. Do NOT imply a working sensor or fabricate an LOD.

## Multiplexing concept [stretch]
Sketch how to detect several analytes at once: orthogonal reporters (different luciferase colors /
FRET pairs), spatial separation (bead/array), or barcoded constructs. Note the cross-talk controls a
multiplex panel needs.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} constructs + off-target/blank controls): $<...>, <...> weeks (IGSC-screened provider).
- Analyte reagent(s) + assay plates + luminometer/plate-reader time: $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
Diagnostic / point-of-care sensing of a disease biomarker (in scope; low dual-use). Gene synthesis via
a biosecurity-screening provider; wet lab under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:600], "...")

## 2 · A simulated dose-response (EXAMPLE_DATA — to design the assay, not to report)

To *plan* the assay you can sketch the expected curve shape from a construct's modeled dynamic range.
This is **EXAMPLE_DATA / SYNTHETIC** — a teaching stand-in for choosing concentrations and replicate
counts, **not** a result. A real curve comes from the lab. We use a simple saturable (Hill) shape.

In [ ]:
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import biosensor_tools as bt

top = pd.read_csv("results/top_constructs.csv") if os.path.exists("results/top_constructs.csv") else pd.DataFrame()
if len(top):
    c = top.iloc[0]
    dr = float(c.get("dynamic_range") or 2.0)
    off = float(c.get("off_signal") or 10.0)
    # EXAMPLE_DATA saturable curve: signal = off * (1 + (dr-1) * x/(x+EC50)); arbitrary EC50.
    EC50 = 5.0   # arbitrary units — the assay measures the real one
    x = np.logspace(-2, 3, 12)
    signal = off * (1.0 + (dr - 1.0) * x / (x + EC50))
    # add a tiny deterministic "assay noise" band for planning replicate counts (SYNTHETIC)
    cv = 0.10
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(x, signal, yerr=cv * signal, fmt="o-", capsize=3)
    ax.set_xscale("log"); ax.set_xlabel("[analyte] (arbitrary units)")
    ax.set_ylabel("signal (a.u., SYNTHETIC)")
    ax.set_title(f"PLANNING dose-response for {c['construct_id'][:24]}... (EXAMPLE_DATA)")
    plt.tight_layout(); plt.savefig("results/p12_doseresponse.png", dpi=150); plt.show()
    lod = bt.estimate_lod(dr, assay_cv=cv)
    print("saved results/p12_doseresponse.png")
    print("LOD PLANNING flag (NOT a measured LOD):", lod)
else:
    print("Run notebook 04 first to produce results/top_constructs.csv.")
print("This curve is EXAMPLE_DATA to size the assay (concentrations, replicates) — never a result.")

## 3 · Off-target specificity controls

The specificity control: the sensor must light up for the **true** analyte and **not** for a related
or unrelated protein. Here we scaffold the panel deterministically; on Colab, run the integrated
construct against each off-target analyte (two-state / AF2) and confirm the dynamic range collapses.

In [ ]:
import biosensor_tools as bt

# Define an off-target panel (EXAMPLE — choose real related/unrelated proteins for your analyte).
OFF_TARGETS = ["RELATED_BIOMARKER_X", "UNRELATED_PROTEIN_Y"]
rows = []
if len(top):
    for _, c in top.head(5).iterrows():
        for ot in OFF_TARGETS:
            # On Colab: re-model the construct with the OFF-TARGET present; dynamic range SHOULD ~1.
            # Mock proxy: an off-target should not switch -> force a near-1 dynamic range (SYNTHETIC).
            rows.append(dict(construct_id=c["construct_id"], off_target=ot,
                             expected_dynamic_range=1.0,
                             note="off-target must NOT switch (specificity control) — SYNTHETIC proxy"))
    pd.DataFrame(rows).to_csv("results/offtarget_controls.csv", index=False)
    print(f"wrote results/offtarget_controls.csv: {len(rows)} off-target control rows (SYNTHETIC)")
    print("On Colab: re-run two-state modeling with each off-target; confirm dynamic_range ~ 1 (no switching).")
else:
    print("Run notebook 04 first to produce results/top_constructs.csv.")

## 4 · (Stretch) Multiplexing concept `[stretch]`

Detecting several analytes at once needs **orthogonal** reporters (different luciferase colors / FRET
pairs), spatial separation (bead/array), or barcoded constructs — plus **cross-talk** controls
(each sensor tested against the others' analytes). Sketch the panel; do not over-claim. This is a
*concept* deliverable, not measured data.

In [ ]:
multiplex_note = """Multiplexing concept (stretch) — sketch, not data:
- Orthogonal readouts: pair each analyte's sensor with a distinguishable reporter (e.g. different
  luciferase substrates/colors, or distinct FRET donor/acceptor pairs).
- Spatial encoding: immobilize sensors on addressable beads/array spots (one analyte per spot).
- Cross-talk controls (MANDATORY): test every sensor against every other analyte; quantify spillover.
- Decide read-out scheme (ratiometric / spectral unmixing) and the blank+off-target controls per channel.
"""
open("results/multiplexing_concept.md", "w").write(multiplex_note)
print("wrote results/multiplexing_concept.md (concept only — no fabricated multiplex data).")

## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: luminescence/FRET **dose-response**, LOD *estimate from data*, expression, timeline, costed reagents.
- [ ] Controls specified: **no-analyte / blank** (OFF floor + LOD) and **off-target** (`results/offtarget_controls.csv`); positive/calibrator.
- [ ] Dose-response *planning* curve is labeled EXAMPLE_DATA; **no fabricated LOD/luminescence** reported as real.
- [ ] (Stretch) multiplexing concept with cross-talk controls (`results/multiplexing_concept.md`).
- [ ] Honest framing: coupling binding to a clean ON/OFF is hard; report measured dynamic range + LOD, including failures.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — an integrated **binder + switch** biosensor with a functional-readout plan, built on the
binder-family workflow.